# Eksperimen 12: Enhanced Anchor LGBM
**submission_09.csv | Public Score: 1.642**

Eksperimen ini berhasil melampaui batas *Hard Ceiling* 1.65 yang ditemui oleh pendekatan murni cuaca (*Exogenous-Only*) dari eksperimen sebelumnya. Kunci terobosan ini adalah **Fitur Anchor TMA** — kondisi nyata tinggi air sungai di beberapa titik waktu sebelum batas akhir data latih (cutoff), yang digabungkan dengan sinyal peluruhan eksponensial untuk merepresentasikan bagaimana relevansi informasi TMA masa lalu memudar seiring bertambahnya jarak prediksi ke masa depan.

**Temuan Penting:** CV lokal menunjukkan hasil lebih buruk (1.72) dibandingkan Eksperimen 10 (1.71) karena fitur anchor bersifat statis per-pos selama training. Namun Public Score justru membaik ke **1.642**, membuktikan kondisi sungai nyata di akhir September sangat informatif untuk prediksi Oktober–Maret.

In [ ]:
import pandas as pd
import numpy as np
import warnings
import lightgbm as lgb
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error
from sklearn.cluster import KMeans
from sklearn.model_selection import TimeSeriesSplit
import optuna
import os

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 150)
optuna.logging.set_verbosity(optuna.logging.WARNING)

## 1. Pemuatan Data

In [ ]:
train = pd.read_csv('../data/raw/train.csv')
test = pd.read_csv('../data/raw/test.csv')
env_data = pd.read_csv('../data/raw/data_pendukung/data_lingkungan.csv')
coords = pd.read_csv('../data/raw/data_pendukung/koordinat_pos.csv')

train['datetime'] = pd.to_datetime(train['datetime'])
test['datetime'] = pd.to_datetime(test['id'].str[:19])
test['nama_pos'] = test['id'].str[22:]
env_data['datetime'] = pd.to_datetime(env_data['datetime'])

overall_cutoff = train['datetime'].max()
print("Batas Akhir Train:", overall_cutoff)
print("Awal Test        :", test['datetime'].min())

## 2. EDA & CEDA

In [ ]:
print("=== Dimensi Matriks ===")
print("Train:", train.shape)
print("Test:", test.shape)
print("Lingkungan:", env_data.shape)

print("\n=== Missing Values Data Lingkungan ===")
print(env_data.isnull().sum()[env_data.isnull().sum() > 0])

### 2.1 CEDA: Distribusi TMA & Analisis Kondisi Sungai Menjelang Cutoff

In [ ]:
station_stats = train.groupby('nama_pos')['tma_mdpl'].agg(['mean', 'std', 'min', 'max'])
print("=== Distribusi TMA per Pos ===")
print(station_stats.sort_values('mean', ascending=False).round(2).to_string())

## 3. Station Normalization Profile

In [ ]:
global_mean = train['tma_mdpl'].mean()
global_std = train['tma_mdpl'].std()

station_profile = train.groupby('nama_pos')['tma_mdpl'].agg(
    tma_mean='mean',
    tma_std='std',
    tma_p25=lambda x: x.quantile(0.25),
    tma_p75=lambda x: x.quantile(0.75)
).reset_index()
station_profile['tma_std'] = station_profile['tma_std'].fillna(1.0)

## 4. Enhanced Anchor TMA Features (Anti-Leakage)
Inovasi utama eksperimen ini. Untuk setiap pos, kita merekam kondisi TMA di beberapa titik waktu sebelum cutoff dari data train. Semua fitur ini **tidak mengandung kebocoran data** (*data leakage*) karena:
1. Nilainya diambil murni dari data train.
2. Tidak ada nilai dari data test yang digunakan.

Fitur `anchor_decay_signal` merupakan fitur paling kreatif: `tma_anchor_0h × exp(-max(steps_ahead, 0) / 56)`. Untuk data test, nilainya meluruh secara eksponensial seiring bertambahnya jarak prediksi. Untuk data train (steps_ahead negatif), nilainya sama dengan `tma_anchor_0h`.

In [ ]:
train_sorted = train.sort_values(['nama_pos', 'datetime'])

def get_anchor_tma(pos_df, target_dt):
    subset = pos_df[pos_df['datetime'] <= target_dt]
    if subset.empty:
        return np.nan
    idx = (subset['datetime'] - target_dt).abs().idxmin()
    return subset.loc[idx, 'tma_mdpl']

def get_tma_window_stats(pos_df, end_dt, days):
    start_dt = end_dt - pd.Timedelta(f'{days}D')
    subset = pos_df[(pos_df['datetime'] >= start_dt) & (pos_df['datetime'] <= end_dt)]['tma_mdpl']
    if len(subset) < 2:
        return np.nan, np.nan
    x = np.arange(len(subset))
    slope = np.polyfit(x, subset.values, 1)[0]
    return subset.std(), slope

anchor_records = []
for pos in train['nama_pos'].unique():
    pos_df = train_sorted[train_sorted['nama_pos'] == pos]
    tma_0h   = get_anchor_tma(pos_df, overall_cutoff)
    tma_24h  = get_anchor_tma(pos_df, overall_cutoff - pd.Timedelta('24H'))
    tma_72h  = get_anchor_tma(pos_df, overall_cutoff - pd.Timedelta('72H'))
    tma_168h = get_anchor_tma(pos_df, overall_cutoff - pd.Timedelta('168H'))
    tma_336h = get_anchor_tma(pos_df, overall_cutoff - pd.Timedelta('336H'))
    tma_vol_7d,  tma_trend_7d  = get_tma_window_stats(pos_df, overall_cutoff, 7)
    tma_vol_30d, tma_trend_30d = get_tma_window_stats(pos_df, overall_cutoff, 30)

    anchor_records.append({
        'nama_pos': pos,
        'tma_anchor_0h': tma_0h,
        'tma_anchor_24h': tma_24h,
        'tma_anchor_72h': tma_72h,
        'tma_anchor_168h': tma_168h,
        'tma_anchor_336h': tma_336h,
        'tma_volatility_7d': tma_vol_7d,
        'tma_trend_7d': tma_trend_7d,
        'tma_volatility_30d': tma_vol_30d,
        'tma_trend_30d': tma_trend_30d,
    })

anchor_df = pd.DataFrame(anchor_records)
station_profile = pd.merge(station_profile, anchor_df, on='nama_pos', how='left')
print("Preview Anchor Features:")
print(anchor_df[['nama_pos', 'tma_anchor_0h', 'tma_anchor_168h', 'tma_trend_7d', 'tma_volatility_7d']].to_string())

## 5. Preprocessing Adaptif

In [ ]:
env_data = env_data.sort_values(['nama_pos', 'datetime'])
macro_cols = ['nino_34', 'mjo_phase', 'mjo_amplitude', 'mjo_active', 'rmm1', 'rmm2']
dynamic_cols = ['surface_pressure_hpa', 'pressure_msl_hpa', 'soil_moisture_0_7cm',
                'soil_moisture_7_28cm', 'soil_moisture_28_100cm', 'soil_moisture_100_255cm']

for c in macro_cols:
    env_data[c] = env_data.groupby('nama_pos')[c].ffill().bfill()

for c in dynamic_cols:
    env_data[c] = env_data.groupby('nama_pos')[c].apply(
        lambda x: x.interpolate(method='linear').bfill().ffill()
    ).reset_index(level=0, drop=True)

### 5.1 Agregasi & Penggabungan Spasial

In [ ]:
kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
coords['spatial_cluster'] = kmeans.fit_predict(coords[['latitude', 'longitude']])

def aggregate_env_data(df):
    agg_funcs = {col: 'mean' for col in df.columns if col not in ['nama_pos', 'landcover_name', 'datetime']}
    agg_funcs['rainfall_mm'] = 'sum'
    agg_funcs['rainfall_openmeteo_mm'] = 'sum'
    agg_funcs['rainfall_max_24h_mm'] = 'max'
    df_indexed = df.set_index('datetime')
    return df_indexed.groupby(['nama_pos', pd.Grouper(freq='3h', label='right', closed='right')]).agg(agg_funcs).reset_index()

env_agg = aggregate_env_data(env_data)

test['tma_mdpl'] = np.nan
all_data = pd.concat([train, test], ignore_index=True)
all_data = all_data.sort_values(['nama_pos', 'datetime']).reset_index(drop=True)

all_data = pd.merge(all_data, env_agg, on=['datetime', 'nama_pos'], how='left')
all_data = pd.merge(all_data, coords, on='nama_pos', how='left')
all_data = pd.merge(all_data, station_profile, on='nama_pos', how='left')

all_data['tma_mean'] = all_data['tma_mean'].fillna(global_mean)
all_data['tma_std'] = all_data['tma_std'].fillna(global_std)

## 6. Deep Feature Engineering

In [ ]:
le = LabelEncoder()
all_data['nama_pos_encoded'] = le.fit_transform(all_data['nama_pos'])

all_data['month'] = all_data['datetime'].dt.month
all_data['hour'] = all_data['datetime'].dt.hour
all_data['day_of_year'] = all_data['datetime'].dt.dayofyear
all_data['sin_hour'] = np.sin(2 * np.pi * all_data['hour'] / 24)
all_data['cos_hour'] = np.cos(2 * np.pi * all_data['hour'] / 24)
all_data['sin_month'] = np.sin(2 * np.pi * all_data['month'] / 12)
all_data['cos_month'] = np.cos(2 * np.pi * all_data['month'] / 12)

all_data['steps_ahead'] = (all_data['datetime'] - overall_cutoff) / pd.Timedelta('3H')
all_data['hours_ahead'] = all_data['steps_ahead'] * 3

for col in ['tma_anchor_0h', 'tma_anchor_24h', 'tma_anchor_72h', 'tma_anchor_168h', 'tma_anchor_336h']:
    col_norm = col + '_norm'
    all_data[col_norm] = (all_data[col] - all_data['tma_mean']) / all_data['tma_std']

all_data['anchor_decay_signal'] = (
    all_data['tma_anchor_0h_norm'] * np.exp(-np.clip(all_data['steps_ahead'], 0, None) / 56)
)
all_data['anchor_trend_signal'] = (
    all_data['tma_trend_7d'] * np.exp(-np.clip(all_data['steps_ahead'], 0, None) / 112)
)

all_data['runoff_factor'] = all_data['rainfall_mm'] * all_data['soil_moisture_0_7cm']
all_data['pressure_drop'] = all_data.groupby('nama_pos')['surface_pressure_hpa'].diff(1).fillna(0)
all_data['temp_humidity_index'] = all_data['temperature_c'] * all_data['humidity_pct'] / 100

windows = [4, 8, 24, 56]
for w in windows:
    all_data[f'rainfall_roll_{w}'] = all_data.groupby('nama_pos')['rainfall_mm'].transform(
        lambda x: x.rolling(window=w, min_periods=1).sum()
    )
    all_data[f'soil_roll_{w}'] = all_data.groupby('nama_pos')['soil_moisture_0_7cm'].transform(
        lambda x: x.rolling(window=w, min_periods=1).mean()
    )
    all_data[f'pressure_roll_{w}'] = all_data.groupby('nama_pos')['surface_pressure_hpa'].transform(
        lambda x: x.rolling(window=w, min_periods=1).mean()
    )
    all_data[f'temp_roll_{w}'] = all_data.groupby('nama_pos')['temperature_c'].transform(
        lambda x: x.rolling(window=w, min_periods=1).mean()
    )

print("Feature engineering selesai.")

## 7. Target Normalization per Pos

In [ ]:
train_mask = all_data['tma_mdpl'].notnull()
all_data['tma_normalized'] = np.nan
all_data.loc[train_mask, 'tma_normalized'] = (
    (all_data.loc[train_mask, 'tma_mdpl'] - all_data.loc[train_mask, 'tma_mean']) /
    all_data.loc[train_mask, 'tma_std']
)

print("Distribusi target ternormalisasi:")
print(all_data.loc[train_mask, 'tma_normalized'].describe().round(4))

## 8. Training Setup

In [ ]:
train_data = all_data[train_mask].sort_values('datetime').reset_index(drop=True)
test_data = all_data[~train_mask].sort_values('datetime').reset_index(drop=True)

drop_cols = ['datetime', 'nama_pos', 'tma_mdpl', 'tma_normalized', 'id', 'landcover_name',
             'tma_anchor_0h', 'tma_anchor_24h', 'tma_anchor_72h', 'tma_anchor_168h', 'tma_anchor_336h']
features = [c for c in train_data.columns if c not in drop_cols]
target = 'tma_normalized'

X_full = train_data[features]
y_full = train_data[target]
X_test = test_data[features]

tscv = TimeSeriesSplit(n_splits=5)
print(f"Total fitur: {len(features)}")
print(features)

### 8.1 Optuna Tuning: Max Power LGBM (45 Trials)

In [ ]:
def objective_lgb(trial):
    params = {
        'n_estimators': 3000,
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.05, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 31, 255),
        'max_depth': trial.suggest_int('max_depth', 5, 12),
        'subsample': trial.suggest_float('subsample', 0.5, 0.9),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 0.9),
        'min_child_samples': trial.suggest_int('min_child_samples', 30, 300),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.1, 20.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.1, 20.0, log=True),
        'subsample_freq': 1, 'random_state': 42, 'verbose': -1
    }
    scores = []
    for train_idx, val_idx in tscv.split(X_full):
        X_tr, X_va = X_full.iloc[train_idx], X_full.iloc[val_idx]
        y_tr, y_va = y_full.iloc[train_idx], y_full.iloc[val_idx]
        m = lgb.LGBMRegressor(**params)
        m.fit(X_tr, y_tr, eval_set=[(X_va, y_va)],
              callbacks=[lgb.early_stopping(200, verbose=False)])
        scores.append(mean_squared_error(y_va, m.predict(X_va)))
    return np.mean(scores)

print("Memulai Optuna Tuning (45 trials)...")
study = optuna.create_study(direction='minimize')
study.optimize(objective_lgb, n_trials=45)
best_lgb = study.best_params
best_lgb.update({'n_estimators': 3000, 'random_state': 42, 'verbose': -1, 'subsample_freq': 1})
print(f"Best Params: {best_lgb}")

## 9. Final K-Fold Training

In [ ]:
test_preds = np.zeros(len(X_test))
cv_scores = []

print("Memulai K-Fold Final Training...")
for fold, (train_idx, val_idx) in enumerate(tscv.split(X_full)):
    X_tr, X_va = X_full.iloc[train_idx], X_full.iloc[val_idx]
    y_tr, y_va = y_full.iloc[train_idx], y_full.iloc[val_idx]

    val_mean = train_data.iloc[val_idx]['tma_mean'].values
    val_std = train_data.iloc[val_idx]['tma_std'].values

    m = lgb.LGBMRegressor(**best_lgb)
    m.fit(X_tr, y_tr, eval_set=[(X_va, y_va)],
          callbacks=[lgb.early_stopping(200, verbose=False)])

    p_norm = m.predict(X_va)
    p_abs = (p_norm * val_std) + val_mean
    y_abs = (y_va.values * val_std) + val_mean
    fold_rmse = np.sqrt(mean_squared_error(y_abs, p_abs))
    cv_scores.append(fold_rmse)
    print(f"Fold {fold+1} RMSE (abs): {fold_rmse:.4f} | Trees Matang: {m.best_iteration_}")
    test_preds += m.predict(X_test) / tscv.n_splits

print(f"\nRata-rata K-Fold RMSE (Enhanced Anchor LGBM): {np.mean(cv_scores):.4f}")

## 10. Denormalisasi & Output Submisi

In [ ]:
test_mean = test_data['tma_mean'].values
test_std = test_data['tma_std'].values
final_tma = (test_preds * test_std) + test_mean

test_data['tma_mdpl'] = final_tma
submission = test_data[['id', 'tma_mdpl']]

if not os.path.exists('../submissions'):
    os.makedirs('../submissions')

submission.to_csv('../submissions/submission_09.csv', index=False)
print("Eksperimen 12 selesai. File submission_09.csv tersimpan.")
print(f"Rentang prediksi TMA: {final_tma.min():.2f} - {final_tma.max():.2f}")